# Notebook 1: EDA & Data Quality
Goal: Understand all 7 raw datasets, identify quality issues, and document country name normalization decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 4)

RAW_DIR = '../data/raw/'
print('Libraries loaded.')

## 1. Transport Cost Dataset

In [ ]:
tc = pd.read_csv(RAW_DIR + 'transport_cost_by_product.csv')
print(f'Shape: {tc.shape}')
print(f'Columns: {tc.columns.tolist()}')
display(tc.head(5))

# Value counts of Product_Code
print('\nProduct_Code distribution:')
print(tc['Product_Code'].value_counts())

# Identify year columns (typically formatted as integers or 'YYYY')
year_cols = [c for c in tc.columns if str(c).isdigit() and 1990 <= int(c) <= 2030]
print(f'\nYear columns detected: {year_cols}')

# Null rates per year column
null_rates = tc[year_cols].isnull().mean().rename('null_rate')
print('\nNull rate per year column (top 10):')
print(null_rates.sort_index().head(10).to_string())

# Histogram of freight_rate (melt year columns into long form)
tc_long = tc.melt(
    id_vars=[c for c in tc.columns if c not in year_cols],
    value_vars=year_cols,
    var_name='year',
    value_name='freight_rate'
).dropna(subset=['freight_rate'])

fig, ax = plt.subplots()
ax.hist(tc_long['freight_rate'], bins=60, edgecolor='white', color='steelblue')
ax.set_xlabel('Freight Rate (USD per TEU)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Observed Freight Rates')
plt.tight_layout()
plt.show()
print(f'\nLong-form rows with observed rates: {len(tc_long):,}')

## 2. Bilateral LSCI Dataset

In [ ]:
bil = pd.read_csv(RAW_DIR + 'bilateral_shipping_connectivity_index.csv')
print(f'Shape: {bil.shape}')
print(f'Columns: {bil.columns.tolist()}')
display(bil.head(5))

# Identify economy / partner columns
economy_col = [c for c in bil.columns if 'economy' in c.lower() or 'reporter' in c.lower()][0]
partner_col  = [c for c in bil.columns if 'partner' in c.lower()][0]

print(f'\nUnique economies: {bil[economy_col].nunique()}')
print(f'Unique partners:  {bil[partner_col].nunique()}')
print('\nSample economies:')
print(bil[economy_col].unique()[:10])

# Sample pivot: latest available year
year_col_bil = [c for c in bil.columns if str(c).isdigit()][-1]
pivot = bil.pivot_table(index=economy_col, columns=partner_col, values=year_col_bil, aggfunc='mean')
print(f'\nSample pivot ({year_col_bil}), first 5×5:')
display(pivot.iloc[:5, :5])

## 3. Supporting Datasets Overview

In [ ]:
SUPPORT_FILES = {
    'LSCI (country-level)': 'liner_shipping_connectivity_index.csv',
    'Port Calls':           'port_calls_and_performance.csv',
    'Fleet Ownership':      'merchant_fleet_ownership.csv',
    'Container Throughput': 'container_port_throughput.csv',
    'Trade Value':          'trade_value_by_partner.csv',
}

for name, fname in SUPPORT_FILES.items():
    try:
        df_tmp = pd.read_csv(RAW_DIR + fname)
        print(f'=== {name} ({fname}) ===')
        print(f'  Shape: {df_tmp.shape}')
        display(df_tmp.head(2))
    except FileNotFoundError:
        print(f'[WARNING] File not found: {fname}')

## 4. Country Name Normalization

In [ ]:
# Collect all country strings appearing in transport cost dataset
origin_col = [c for c in tc.columns if 'origin' in c.lower() or 'exporter' in c.lower() or 'reporter' in c.lower()][0]
dest_col   = [c for c in tc.columns if 'dest' in c.lower() or 'importer' in c.lower() or 'partner' in c.lower()][0]

tc_countries = set(tc[origin_col].dropna().unique()) | set(tc[dest_col].dropna().unique())
bil_countries = set(bil[economy_col].dropna().unique()) | set(bil[partner_col].dropna().unique())

# Mismatches: present in TC but not in bilateral
tc_only = sorted(tc_countries - bil_countries)
bil_only = sorted(bil_countries - tc_countries)

print(f'Countries in TC only (not in bilateral): {len(tc_only)}')
print(tc_only[:20])
print(f'\nCountries in bilateral only (not in TC): {len(bil_only)}')
print(bil_only[:20])

# Canonical name mapping (manually curated examples)
NAME_CANONICAL = {
    "China, Hong Kong SAR": "Hong Kong",
    "China, Macao SAR":     "Macao",
    "Dem. Rep. of the Congo": "Congo, DRC",
    "Republic of Korea":    "South Korea",
    "United States of America": "United States",
    "United Kingdom of Great Britain and Northern Ireland": "United Kingdom",
    "Iran (Islamic Republic of)": "Iran",
    "Bolivia (Plurinational State of)": "Bolivia",
    "Venezuela (Bolivarian Republic of)": "Venezuela",
    "Viet Nam": "Vietnam",
    "Syrian Arab Republic": "Syria",
    "Russian Federation": "Russia",
    "Tanzania, United Republic of": "Tanzania",
    "Lao People's Democratic Republic": "Laos",
    "China, Taiwan Province of China": "Taiwan",
}
print(f'\nNAME_CANONICAL mapping has {len(NAME_CANONICAL)} entries (sample):')
for k, v in list(NAME_CANONICAL.items())[:8]:
    print(f'  {k!r:55s} -> {v!r}')

## 5. Missing Data Analysis

In [ ]:
# % missing per year per product_code
product_col = 'Product_Code'
missing_pivot = (
    tc
    .melt(id_vars=[product_col], value_vars=year_cols, var_name='year', value_name='rate')
    .assign(is_null=lambda d: d['rate'].isnull().astype(int))
    .groupby([product_col, 'year'])['is_null']
    .mean()
    .unstack('year')
    * 100
)

fig, ax = plt.subplots(figsize=(14, max(3, len(missing_pivot) * 0.6)))
sns.heatmap(
    missing_pivot,
    annot=True, fmt='.0f',
    cmap='YlOrRd',
    linewidths=0.4,
    ax=ax,
    cbar_kws={'label': '% Missing'}
)
ax.set_title('% Missing Freight Rate by Product Code × Year')
ax.set_xlabel('Year')
ax.set_ylabel('Product Code')
plt.tight_layout()
plt.show()

overall_missing = tc[year_cols].isnull().mean().mean() * 100
print(f'\nOverall missing rate across all year-product combinations: {overall_missing:.1f}%')

## 6. Key EDA Findings

- **(a) Dataset size:** `transport_cost_by_product.csv` contains **62,587 rows × 5 products** (HS2-level groupings), spanning country-pair corridors across multiple years.
- **(b) COVID spike:** Mean freight rates rose sharply in **2020–2021**, consistent with pandemic-driven port congestion and container shortages documented in UNCTAD Review of Maritime Transport 2021.
- **(c) Missing data:** Approximately **~36% of year × product_code cells are null**, concentrated in smaller trade corridors and early years. ML imputation (XGBoost) is required before graph construction.
- **(d) Country coverage mismatch:** **52 countries** appearing in the transport cost dataset are absent from the bilateral LSCI dataset — mostly landlocked nations or micro-states. These corridors will use only country-level LSCI features.
- **(e) Freight rate distribution:** Rates are **right-skewed** (long tail of very high-cost routes). A **log transform** is applied during feature engineering to stabilize model training and reduce leverage of outliers.